In [0]:
# import dlt
# from pyspark.sql.functions import col, from_json, current_timestamp, lit
# from pyspark.sql.types import (
#     StructType, StructField, StringType, IntegerType, DecimalType, TimestampType
# )

# order_schema = StructType([
#     StructField("order_id", StringType(), True),
#     StructField("total_amount", DecimalType(), True),
#     StructField("product_id", IntegerType(), True),
#     StructField("customer_id", IntegerType(), True),
#     StructField("store_id", IntegerType(), True),
#     StructField("quantity", IntegerType(), True),
#     StructField("event_time", TimestampType(), True),
# ])

# inventory_schema = StructType([
#     StructField("current_stock", IntegerType(), True),
#     StructField("quantity_change", IntegerType(), True),
#     StructField("change_type", StringType(), True),
#     StructField("product_id", IntegerType(), True),
#     StructField("store_id", IntegerType(), True),
#     StructField("event_time", TimestampType(), True),
# ])

In [0]:
# @dlt.table(
#     name="maven_catalog.bronze_schema.brz_kafka_orders_raw",
#     comment="Bronze table sourced from Kafka's landing table",
#     table_properties={
#         "quality": "bronze", 
#     }
# )
# def brz_kafka_orders_raw():
#     return (
#         spark.readStream
#         .option("maxFilesPerTrigger", 1) 
#         .table("maven_catalog.maven_market_landing.maven_orders")
#     )

In [0]:
# @dlt.table(
#     name="maven_catalog.bronze_schema.brz_kafka_orders",
#     comment="Orders parsed by Fivetran from Kafka",
#     table_properties={"quality": "bronze"}
# )
# @dlt.expect("valid_product_id", "product_id IS NOT NULL")
# @dlt.expect("valid_customer_id", "customer_id IS NOT NULL")
# @dlt.expect("valid_store_id", "store_id IS NOT NULL")
# @dlt.expect("valid_quantity", "quantity > 0")
# @dlt.expect("valid_event_time", "event_time IS NOT NULL")
# def brz_kafka_orders():
#     return (
#         dlt.read_stream("maven_catalog.bronze_schema.brz_kafka_orders_raw")
#         .select(
#             col("value_order_id").alias("order_id"),
#             col("value_total_amount").cast("decimal(18,2)").alias("total_amount"),
#             col("value_product_id").alias("product_id"),
#             col("value_customer_id").alias("customer_id"),
#             col("value_store_id").alias("store_id"),
#             col("value_quantity").alias("quantity"),
#             col("value_event_time").alias("event_time"),
#             # Including Fivetran metadata for auditing
#             col("_fivetran_synced").alias("_ingested_by_fivetran_at")
#         )
#         .withColumn("_ingestion_timestamp", current_timestamp())
#         .withColumn("_source_system", lit("fivetran_kafka_inventory"))
#     )

In [0]:
# @dlt.table(
#     name="maven_catalog.bronze_schema.brz_kafka_inventory_raw",
#     comment="Incremental stream from Fivetran landing table for inventory",
#     table_properties={"quality": "bronze", "source": "fivetran_landing"}
# )
# def brz_kafka_inventory_raw():
#     # Replace with the actual catalog.schema.table name created by Fivetran
#     return (
#         spark.readStream
#         .table("maven_catalog.maven_market_landing.maven_inventory")
#     )

In [0]:
# @dlt.table(
#     name="maven_catalog.bronze_schema.brz_kafka_inventory",
#     comment="Inventory events unpacked from Fivetran landing table",
#     table_properties={"quality": "bronze"}
# )
# @dlt.expect("valid_product_id", "product_id IS NOT NULL")
# @dlt.expect("valid_store_id", "store_id IS NOT NULL")
# @dlt.expect("valid_stock_level", "current_stock >= 0")
# @dlt.expect("valid_event_time", "event_time IS NOT NULL")
# def brz_kafka_inventory():
#     return (
#         dlt.read_stream("maven_catalog.bronze_schema.brz_kafka_inventory_raw")
#         .select(
#             col("value_current_stock").alias("current_stock"),
#             col("value_quantity_change").alias("quantity_change"),
#             col("value_change_type").alias("change_type"),
#             col("value_product_id").alias("product_id"),
#             col("value_store_id").alias("store_id"),
#             col("value_event_time").alias("event_time"),
#             # Capture Fivetran sync metadata if needed
#             col("_fivetran_synced").alias("_fivetran_synced")
#         )
#         .withColumn("_ingestion_timestamp", current_timestamp())
#         .withColumn("_source_system", lit("fivetran_kafka_inventory"))
#     )

In [0]:
import dlt
from pyspark.sql.functions import col, from_json, current_timestamp, lit
from pyspark.sql.types import *

# -------------------------
# 1. KAFKA CONNECTION CONFIG
# -------------------------
# Assuming your secrets are stored in a scope named "maven-scope"
kafka_bootstrap_servers = dbutils.secrets.get(scope="maven-secret-scopes", key="kafka-bootstrap-servers")
kafka_api_key = dbutils.secrets.get(scope="maven-secret-scopes", key="kafka-api-key")
kafka_api_secret = dbutils.secrets.get(scope="maven-secret-scopes", key="kafka-api-secret")

# Confluent Cloud Security Protocol
kafka_options = {
    "kafka.bootstrap.servers": kafka_bootstrap_servers,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{kafka_api_key}' password='{kafka_api_secret}';",
    "subscribe": "", # This will be set per table
    "startingOffsets": "earliest"
}

In [0]:
# -------------------------
# 2. SCHEMAS
# -------------------------
# Note: Changed DecimalType to Double/Float to match Python's JSON float output
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("event_time", StringType(), True), # ISO format comes as string
    StructField("store_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("total_amount", DoubleType(), True)
])

inventory_schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("change_type", StringType(), True),
    StructField("quantity_change", IntegerType(), True),
    StructField("current_stock", IntegerType(), True)
])

In [0]:
# -------------------------
# 3. BRONZE TABLES (DIRECT KAFKA)
# -------------------------

@dlt.table(
    name="maven_catalog.bronze_schema.brz_kafka_orders",
    comment="Direct stream from Confluent Kafka orders topic",
    table_properties={"quality": "bronze"}
)
@dlt.expect("valid_order_id", "order_id IS NOT NULL")
def brz_kafka_orders():
    # Set topic for this specific stream
    options = kafka_options.copy()
    options["subscribe"] = "maven.orders"
    
    return (
        spark.readStream
        .format("kafka")
        .options(**options)
        .load()
        # Kafka data is in 'value' column as binary
        .select(from_json(col("value").cast("string"), order_schema).alias("data"))
        .select("data.*")
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit("confluent_kafka_direct"))
    )

@dlt.table(
    name="maven_catalog.bronze_schema.brz_kafka_inventory",
    comment="Direct stream from Confluent Kafka inventory topic",
    table_properties={"quality": "bronze"}
)
@dlt.expect("valid_store_id", "store_id IS NOT NULL")
def brz_kafka_inventory():
    options = kafka_options.copy()
    options["subscribe"] = "maven.inventory"
    
    return (
        spark.readStream
        .format("kafka")
        .options(**options)
        .load()
        .select(from_json(col("value").cast("string"), inventory_schema).alias("data"))
        .select("data.*")
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit("confluent_kafka_direct"))
    )